In [1]:
import asyncio
import anthropic
import time

In [3]:
# NOTE: In Jupyter, run the last line as:  await research_agent("your topic")
# If running as a .py script, use:         asyncio.run(research_agent("your topic"))

# Set your API key: 
import os; 
os.environ["ANTHROPIC_API_KEY"] = "your api key here"
aclient = anthropic.AsyncAnthropic()

In [4]:
async def research_topic(topic: str, angle: str) -> dict:
    r = await aclient.messages.create(
        model="claude-sonnet-4-5", max_tokens=300,
        system=f"You are a {angle} analyst. Be concise and specific.",
        messages=[{"role": "user", "content":
            f"Give 3 key insights about: {topic}. Use a numbered list."}])
    return {"angle": angle, "insights": r.content[0].text}


In [5]:
async def synthesize(topic: str, results: list) -> str:
    combined = "\n\n".join([f"{r['angle']}: {r['insights']}" for r in results])
    r = await aclient.messages.create(
        model="claude-sonnet-4-5", max_tokens=500,
        system="Synthesize multi-perspective research into a coherent summary.",
        messages=[{"role": "user", "content":
            f"Topic: {topic}\n\nResearch from multiple angles:\n{combined}\n\nSynthesize into a 3-paragraph summary."}])
    return r.content[0].text

In [6]:
async def research_agent(topic: str) -> str:
    print(f"Researching: {topic}")
    start = time.time()
    results = await asyncio.gather(
        research_topic(topic, "market opportunity"),
        research_topic(topic, "risk and challenge"),
        research_topic(topic, "technology and innovation"),
        return_exceptions=True
    )
    print(f"Parallel research done in {time.time() - start:.1f}s")
    clean = [r for r in results if not isinstance(r, Exception)]
    print(f"Got {len(clean)}/{len(results)} perspectives successfully")
    summary = await synthesize(topic, clean)
    return summary

In [10]:
result = await research_agent("generative AI in Indian enterprises")

print("\n=== SYNTHESIS ===")
print(result)

Researching: generative AI in Indian enterprises
Parallel research done in 7.9s
Got 3/3 perspectives successfully

=== SYNTHESIS ===
# Generative AI in Indian Enterprises: Synthesis

Indian enterprises are rapidly embracing generative AI with a distinct focus on cost optimization and operational efficiency rather than pure innovation. Major IT services giants like TCS, Infosys, and Wipro are embedding AI into their service offerings while developing proprietary platforms, and startups are creating sector-specific solutions for healthcare, agriculture, and fintech. The adoption spans banking, retail, e-commerce, and government sectors, with particular emphasis on automating customer support, code generation, document processing, and testing functions. A unique opportunity exists in developing vernacular AI solutions for India's 22+ official languages, targeting the 90% non-English speaking population. India's 5.4 million-strong IT workforce provides a strategic advantage for implementin